# RD2 Results Pack Builder

This notebook gathers the **final results artifacts** needed for the write-up:

1. QA filtering summaries  
2. Dataset composition table  
3. Matched YOLO detection comparison table  
4. Accepted vs rejected qualitative examples  
5. Exported CSV/PNG files for the paper


z

In [ ]:
# -----------------------
# Section 0 - Imports + config
# -----------------------
from pathlib import Path
import json, textwrap

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
EXPERIMENT_NAME = "PrototypeNine_v1_5"

full_root = BASE_DIR / "output" / EXPERIMENT_NAME
balanced_qa_root = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_vlm_qa_local_v3_balanced"
balanced_filtered_root = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_vlm_filtered_local_v3_balanced"

detect_full_root = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_detect_full"
detect_filtered_root = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_detect_filtered_local_v3_balanced"
matched_metrics_csv = BASE_DIR / "output" / "rd2_detection_comparison_metrics_matched.csv"

results_pack_root = BASE_DIR / "output" / "rd2_results_pack"
fig_root = results_pack_root / "figures"
table_root = results_pack_root / "tables"

for p in [results_pack_root, fig_root, table_root]:
    p.mkdir(parents=True, exist_ok=True)

print("[INFO] results_pack_root:", results_pack_root)


In [ ]:
# -----------------------
# Section 1 - Helpers
# -----------------------
def find_image_for_stem(image_dir: Path, stem: str):
    for ext in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
        p = image_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def summarize_label_counts(label_dir: Path):
    image_count = 0
    box_count = 0
    for p in sorted(label_dir.glob("*.txt")):
        lines = [ln.strip() for ln in p.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            continue
        image_count += 1
        box_count += len(lines)
    return image_count, box_count

def summarize_detect_dataset(root: Path):
    train_images = len(list((root / "images" / "train").glob("*")))
    val_images = len(list((root / "images" / "val").glob("*")))
    _, train_boxes = summarize_label_counts(root / "labels" / "train")
    _, val_boxes = summarize_label_counts(root / "labels" / "val")
    return {
        "train_images": train_images,
        "val_images": val_images,
        "train_boxes": train_boxes,
        "val_boxes": val_boxes,
    }

def wrap_text(s, width=55):
    return "\n".join(textwrap.wrap(str(s), width=width))

def make_side_by_side_example(img_left, title_left, subtitle_left, img_right, title_right, subtitle_right, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, img, title, subtitle in [
        (axes[0], img_left, title_left, subtitle_left),
        (axes[1], img_right, title_right, subtitle_right),
    ]:
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontsize=12, pad=8)
        ax.text(
            0.5, -0.08, wrap_text(subtitle, 45),
            ha="center", va="top", transform=ax.transAxes, fontsize=9
        )
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out_path


In [ ]:
# -----------------------
# Section 2 - Load QA and metrics
# -----------------------
qa_image_csv = balanced_qa_root / "qa_results_per_image.csv"
qa_object_csv = balanced_qa_root / "qa_results_per_object.csv"

assert qa_image_csv.exists(), f"Missing {qa_image_csv}"
assert qa_object_csv.exists(), f"Missing {qa_object_csv}"
assert matched_metrics_csv.exists(), f"Missing {matched_metrics_csv}"

qa_image_df = pd.read_csv(qa_image_csv)
qa_object_df = pd.read_csv(qa_object_csv)
metrics_df = pd.read_csv(matched_metrics_csv)

display(metrics_df)


In [ ]:
# -----------------------
# Section 3 - Dataset composition table
# -----------------------
summary_full = summarize_detect_dataset(detect_full_root)
summary_filtered = summarize_detect_dataset(detect_filtered_root)

dataset_comp_df = pd.DataFrame([
    {
        "dataset": "full_unfiltered",
        "train_images": summary_full["train_images"],
        "val_images": summary_full["val_images"],
        "train_boxes": summary_full["train_boxes"],
        "val_boxes": summary_full["val_boxes"],
    },
    {
        "dataset": "filtered_balanced",
        "train_images": summary_filtered["train_images"],
        "val_images": summary_filtered["val_images"],
        "train_boxes": summary_filtered["train_boxes"],
        "val_boxes": summary_filtered["val_boxes"],
    },
    {
        "dataset": "full_matched_random",
        "train_images": summary_filtered["train_images"],
        "val_images": summary_filtered["val_images"],
        "train_boxes": None,
        "val_boxes": None,
    },
])

dataset_comp_path = table_root / "dataset_composition.csv"
dataset_comp_df.to_csv(dataset_comp_path, index=False)

print("[INFO] Saved:", dataset_comp_path)
display(dataset_comp_df)


In [ ]:
# -----------------------
# Section 4 - QA summary table
# -----------------------
accepted_images = int((qa_image_df["image_verdict"] == "accept").sum())
rejected_images = int((qa_image_df["image_verdict"] != "accept").sum())
total_images = len(qa_image_df)

qa_summary_df = pd.DataFrame([{
    "total_images": total_images,
    "accepted_images": accepted_images,
    "rejected_images": rejected_images,
    "accept_rate_pct": round(100.0 * accepted_images / max(1, total_images), 2),
    "reject_rate_pct": round(100.0 * rejected_images / max(1, total_images), 2),
    "total_objects_judged": len(qa_object_df),
}])

qa_summary_path = table_root / "qa_summary.csv"
qa_summary_df.to_csv(qa_summary_path, index=False)

print("[INFO] Saved:", qa_summary_path)
display(qa_summary_df)

if "defect_type" in qa_object_df.columns:
    defect_counts = qa_object_df["defect_type"].fillna("unknown").value_counts().reset_index()
    defect_counts.columns = ["defect_type", "count"]
    defect_counts_path = table_root / "qa_defect_counts.csv"
    defect_counts.to_csv(defect_counts_path, index=False)
    print("[INFO] Saved:", defect_counts_path)
    display(defect_counts.head(10))


In [ ]:
# -----------------------
# Section 5 - Pick accepted and rejected examples
# -----------------------
accepted_candidates = qa_image_df[qa_image_df["image_verdict"] == "accept"].copy()
accepted_candidates = accepted_candidates.sort_values(["reject_count", "object_count"], ascending=[True, False])

rejected_candidates = qa_image_df[qa_image_df["image_verdict"] != "accept"].copy()
rejected_candidates = rejected_candidates.sort_values(["reject_count", "object_count"], ascending=[False, False])

assert len(accepted_candidates) > 0, "No accepted candidates found"
assert len(rejected_candidates) > 0, "No rejected candidates found"

accepted_row = accepted_candidates.iloc[0]
rejected_row = rejected_candidates.iloc[0]

accepted_stem = accepted_row["image_stem"]
rejected_stem = rejected_row["image_stem"]

accepted_img_path = find_image_for_stem(full_root / "images", accepted_stem)
rejected_img_path = find_image_for_stem(full_root / "images", rejected_stem)

assert accepted_img_path is not None, f"Could not find accepted image for stem {accepted_stem}"
assert rejected_img_path is not None, f"Could not find rejected image for stem {rejected_stem}"

rejected_obj_rows = qa_object_df[qa_object_df["image_stem"] == rejected_stem].copy()

accepted_reason = "Accepted by the balanced VLM QA stage. The image-level verdict was accept, indicating that the generated objects were considered sufficiently reliable for downstream training."

if len(rejected_obj_rows):
    top_reject = rejected_obj_rows.iloc[0]
    rejected_reason = f"Rejected by the balanced VLM QA stage. Example defect type: {top_reject.get('defect_type', 'unknown')}. Reason: {top_reject.get('reason', 'No reason recorded.')}"
else:
    rejected_reason = "Rejected by the balanced VLM QA stage."

display(Image.open(accepted_img_path))
display(Image.open(rejected_img_path))
print("[INFO] accepted_img_path =", accepted_img_path)
print("[INFO] rejected_img_path =", rejected_img_path)
print("[INFO] accepted_reason =", accepted_reason)
print("[INFO] rejected_reason =", rejected_reason)


In [ ]:
# -----------------------
# Section 6 - Export qualitative example figure
# -----------------------
accepted_img = Image.open(accepted_img_path).convert("RGB")
rejected_img = Image.open(rejected_img_path).convert("RGB")

qual_fig_path = fig_root / "accepted_vs_rejected_examples.png"
make_side_by_side_example(
    accepted_img,
    "Accepted example",
    accepted_reason,
    rejected_img,
    "Rejected example",
    rejected_reason,
    qual_fig_path
)

print("[INFO] Saved:", qual_fig_path)
display(Image.open(qual_fig_path))


In [ ]:
# -----------------------
# Section 7 - Export final metrics table
# -----------------------
metrics_out_path = table_root / "matched_detection_metrics.csv"
metrics_df.to_csv(metrics_out_path, index=False)

print("[INFO] Saved:", metrics_out_path)
display(metrics_df)


In [ ]:
# -----------------------
# Section 8 - Build compact text summary
# -----------------------
full_row = metrics_df[metrics_df["dataset_condition"] == "full_unfiltered"].iloc[0]
filtered_row = metrics_df[metrics_df["dataset_condition"] == "filtered_balanced"].iloc[0]
matched_row = metrics_df[metrics_df["dataset_condition"] == "full_matched_random"].iloc[0]

summary_lines = [
    "Results summary for RD2 matched YOLO detection comparison",
    f"- Full unfiltered recall: {full_row['recall_B']:.6f}",
    f"- Filtered balanced recall: {filtered_row['recall_B']:.6f}",
    f"- Full matched random recall: {matched_row['recall_B']:.6f}",
    f"- Full unfiltered mAP50: {full_row['mAP50_B']:.6f}",
    f"- Filtered balanced mAP50: {filtered_row['mAP50_B']:.6f}",
    f"- Full matched random mAP50: {matched_row['mAP50_B']:.6f}",
]

summary_txt = "\n".join(summary_lines)
summary_path = results_pack_root / "results_summary.txt"
summary_path.write_text(summary_txt, encoding="utf-8")

print("[INFO] Saved:", summary_path)
print(summary_txt)
